# scRareBench | High-Level scDML | Dataset 5 — NYGC / Seurat v4 PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook benchmarks **scDML** on registered **Dataset 5**
(`nygc_seurat_v4_pbmc`) through the scRareBench **high-level API**:

`load_dataset(5) → MethodSpec(scDML) → benchmark_method() → scIB + rare-cell metrics → multi-seed HTML / ZIP`

It follows the Dataset 3/4 scDML method-side workflow without manually
implementing dataset preparation, rare-scenario assignment, or benchmark metrics.

### Reproducibility

- **scRareBench:** pinned commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c` (v0.10.6)
- **scDML:** pinned commit `d762dd75557c0dbb2d3039174d69a18195842b93`
- **Method seeds:** `42, 123, 2026`; **benchmark seed:** `42`

### Registered Dataset 5 contract

| Field | Expected value |
|:--|:--|
| Dataset key | `nygc_seurat_v4_pbmc` |
| Source | NYGC / Seurat v4 CITE-seq PBMC |
| Source cells | 161,764 |
| **Analysis cells (after package-owned official QC)** | **152,145** |
| Features | 20,729 |
| Evaluation-only labels | `celltype.l2` (30 categories) |
| Batch variable | `orig.ident` (24 batches) |
| **Canonical raw RNA counts** | **`adata.X`** (`count_layer` is empty/None) |
| scIB reference HVG mode | `evaluation_batch` (on `orig.ident`) |
| Rare metadata | Nine **provisional** registered six-state entries |

**Important:** `load_dataset(5)` already applies the audited official scvi-tools
PBMC QC filter *in memory* and attaches registered rare-scenario annotations.
This notebook must **not** re-filter the data or recompute its scenario taxonomy.
The downloaded source H5AD is left unchanged.

### scDML method policy (same as previous dataset)

- Native scDML preprocessing: 2,000 HVGs, normalize-total `1e4`, log1p,
  batch-wise scaling with cap 10, PCA 50
- Initial Leiden clustering, resolution `3.0`
- `K_in=5`, `K_bw=10`, cosine neighbors, `rule2`
- 50 epochs, hard-triplet loss, Euclidean metric, margin 0.2
- Embedding output 32D, hidden dimension `[256]`
- Fully unsupervised training (`celltype_key=None`, `mode="unsupervised"`)
- Independent scIB reference: **4,000 HVGs**, batch-aware using `orig.ident`

**No biological labels or rare-scenario annotations are used as training
supervision.** `celltype.l2` is reserved for the scRareBench evaluation step.

**Runtime recommendation:** Google Colab GPU + High-RAM. scDML is a legacy
implementation with memory-intensive dense preprocessing/training. All three
seeds are enabled by default, as in the previous notebooks.


### High-level API boundary

`load_dataset(5)` already validates the Dataset 5 source/QC contract, counts,
cell identity, and registered scenario annotations. The notebook does **not**
duplicate those dataset-level checks. The scDML adapter keeps only method-owned
preprocessing/training and exact alignment of the resulting embedding.

## 1. Install the pinned scRareBench and scDML runtimes

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDML_REVISION = "d762dd75557c0dbb2d3039174d69a18195842b93"

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)
SCDML_GITHUB = (
    "git+https://github.com/eleozzr/"
    f"scDML.git@{SCDML_REVISION}"
)

# Bootstrap the exact scRareBench revision without allowing pip to alter
# the preinstalled scientific stack before runtime validation.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

# scDML imports both hnswlib and annoy at import time. These two revisions
# include modern Python support and are pinned for notebook reproducibility.
METHOD_RUNTIME_DEPENDENCIES = (
    "tables>=3.6.1",
    "numpy-groupies>=0.9.14",
    "numexpr>=2.7.1",
    "networkx>=2.5",
    "seaborn>=0.13",
    "torch>=2.0",
    "pytorch-metric-learning==2.9.0",
    "annoy @ git+https://github.com/spotify/annoy.git@379f744667aba6b40ba3db8a07678df173a88f74",
    "hnswlib @ git+https://github.com/nmslib/hnswlib.git@ca426729609b3221563047ceb269cc1213e0d376",
)

from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=(
        "tables",
        "numpy_groupies",
        "numexpr",
        "networkx",
        "seaborn",
        "torch",
        "pytorch_metric_learning",
        "annoy",
        "hnswlib",
    ),
    quiet=False,
)
print_install_report(install_report)

# Install scDML itself without its historical dependency pins. The compatible
# runtime dependencies were installed/validated above.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCDML_GITHUB,
])

# Fresh-process import smoke test catches missing/ABI-incompatible imports now,
# before downloading Dataset 5.
subprocess.check_call([
    sys.executable,
    "-c",
    (
        "import scDML, torch; "
        "from scDML import scDMLModel; "
        "print('scDML', getattr(scDML, '__version__', 'unknown')); "
        "print('torch', torch.__version__)"
    ),
])


## 2. Load Dataset 5 through scRareBench

In [ ]:
from scrarebench import load_dataset

# Dataset 5 QC, metadata, and rare-scenario annotations are package-owned.
adata = load_dataset(5)
print(adata)


## 3. scDML reproducibility and method configuration

These parameters intentionally match the prior scDML high-level notebook.
scDML performs its *own* method-specific preprocessing on raw counts from
**`adata.X`** (not `layers["counts"]`). This is separate from the package-owned
scIB reference pipeline, which uses 4,000 **batch-aware** HVGs.

The method runner preserves the original cell barcodes through scDML's legacy
batch-concatenation behavior and reorders its latent to the benchmark cell order.


In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "cluster_method": "leiden",
    "resolution": 3.0,
    "n_high_var": 2000,
    "normalize_samples": True,
    "target_sum": 1e4,
    "log_normalize": True,
    "normalize_features": True,
    "pca_dim": 50,
    "scale_value": 10.0,

    "K_in": 5,
    "K_bw": 10,
    "K_in_metric": "cosine",
    "K_bw_metric": "cosine",
    "merge_rule": "rule2",

    "num_epochs": 50,
    "projection": False,
    "early_stop": False,
    "batch_size": 64,
    "metric": "euclidean",
    "margin": 0.2,
    "triplet_type": "hard",
    "device": None,  # scDML auto-selects CUDA when available

    "out_dim": 32,
    "emb_dim": [256],
    "save_model": False,
}

OUTPUT_DIR = "/content/scrarebench_scdml_dataset5_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdml_dataset5_method_runs"


## 4. User-owned scDML adapter

The only custom implementation is the scDML method runner: it executes native
scDML preprocessing/training on `adata.X`, keeps `orig.ident` as the batch
covariate (read from the package's dataset profile), and restores original
barcode order if legacy scDML preprocessing changes it.

Dataset source/QC/count/scenario checks already belong to `load_dataset(5)`.
The barcode checks below instead guard the **method's output alignment** before
benchmarking. `benchmark_method()` supplies an isolated AnnData per seed because
scDML mutates its input.


In [ ]:
from pathlib import Path
import gc
import random
import shutil

import numpy as np
import pandas as pd
import torch

from scDML import scDMLModel
from scrarebench import MethodOutput, dataset_info

_ORIGINAL_BARCODE_COL = "_scrarebench_original_barcode"


def run_scdml(method_adata, seed, config):
    # The dataset is already validated and prepared by load_dataset(5).
    # scDML natively reads raw RNA counts from method_adata.X.
    batch_key = dataset_info(method_adata)["batch_key"]

    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)
    # Preserve identity across scDML's legacy AnnData.concatenate() preprocessing.
    method_adata.obs[_ORIGINAL_BARCODE_COL] = original_order

    # scDML.preprocess() consumes adata.X. benchmark_method() supplies a
    # per-seed AnnData copy, so method-side preprocessing cannot overwrite
    # the reference AnnData used for evaluation.

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = scDMLModel(
        verbose=True,
        save_dir=str(seed_dir) + "/",
    )

    adata_scdml = model.preprocess(
        method_adata,
        cluster_method=str(config["cluster_method"]),
        resolution=float(config["resolution"]),
        batch_key=batch_key,
        n_high_var=int(config["n_high_var"]),
        normalize_samples=bool(config["normalize_samples"]),
        target_sum=float(config["target_sum"]),
        log_normalize=bool(config["log_normalize"]),
        normalize_features=bool(config["normalize_features"]),
        pca_dim=int(config["pca_dim"]),
        scale_value=float(config["scale_value"]),
        mode="unsupervised",
    )

    if _ORIGINAL_BARCODE_COL not in adata_scdml.obs.columns:
        raise RuntimeError(
            "Original cell-identity guard was lost during scDML preprocessing."
        )
    if "init_cluster" not in adata_scdml.obs.columns:
        raise RuntimeError("scDML preprocessing did not create init_cluster.")

    num_init_clusters = int(adata_scdml.obs["init_cluster"].nunique())
    if num_init_clusters < 2:
        raise RuntimeError(
            f"scDML produced too few initial clusters: {num_init_clusters}"
        )

    # expect_num_cluster=None makes scDML estimate the target cluster count by
    # its eigengap heuristic. All candidate merge counts are therefore prepared.
    ncluster_candidates = list(range(2, num_init_clusters + 1))

    print(
        f"Seed {seed}: cells={adata_scdml.n_obs:,}, "
        f"features={adata_scdml.n_vars:,}, "
        f"batches={adata_scdml.obs[batch_key].nunique()}, "
        f"initial Leiden clusters={num_init_clusters}"
    )

    model.integrate(
        adata_scdml,
        batch_key=batch_key,
        ncluster_list=ncluster_candidates,
        expect_num_cluster=None,
        K_in=int(config["K_in"]),
        K_bw=int(config["K_bw"]),
        K_in_metric=str(config["K_in_metric"]),
        K_bw_metric=str(config["K_bw_metric"]),
        merge_rule=str(config["merge_rule"]),
        num_epochs=int(config["num_epochs"]),
        projection=bool(config["projection"]),
        early_stop=bool(config["early_stop"]),
        batch_size=int(config["batch_size"]),
        metric=str(config["metric"]),
        margin=float(config["margin"]),
        triplet_type=str(config["triplet_type"]),
        device=config["device"],
        seed=int(seed),
        out_dim=int(config["out_dim"]),
        emb_dim=list(config["emb_dim"]),
        save_model=bool(config["save_model"]),
        celltype_key=None,
        mode="unsupervised",
    )

    if "X_emb" not in adata_scdml.obsm:
        raise RuntimeError("scDML did not create adata.obsm['X_emb'].")

    latent_native = np.asarray(
        adata_scdml.obsm["X_emb"],
        dtype=np.float32,
    )
    native_barcodes = (
        adata_scdml.obs[_ORIGINAL_BARCODE_COL]
        .astype(str)
        .to_numpy()
    )

    if latent_native.ndim != 2:
        raise RuntimeError(
            f"scDML latent must be 2D; observed shape={latent_native.shape}."
        )
    if latent_native.shape[0] != len(native_barcodes):
        raise RuntimeError(
            "scDML latent rows do not match the post-preprocessing cells."
        )
    if not np.isfinite(latent_native).all():
        raise ValueError("scDML latent contains NaN or infinite values.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError(
            "Cell identifiers are no longer unique after scDML preprocessing."
        )
    if set(native_barcodes) != set(original_order):
        raise RuntimeError(
            "scDML preprocessing changed the benchmark cell set."
        )

    # Restore the exact original scRareBench row order.
    position = pd.Series(
        np.arange(len(native_barcodes)),
        index=native_barcodes,
    )
    reorder = position.loc[original_order].to_numpy()
    latent = latent_native[reorder]

    if not np.array_equal(native_barcodes[reorder], original_order):
        raise RuntimeError(
            "Failed to restore scDML latent to the original benchmark order."
        )

    provenance = {}
    log_path = seed_dir / "log.txt"
    if log_path.exists() and log_path.stat().st_size > 0:
        provenance["scdml_log"] = log_path

    print(f"Seed {seed}: aligned scDML latent shape={latent.shape}")

    del model, adata_scdml, latent_native, position, reorder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdml",
        provenance_files=provenance,
    )


## 5. Run scRareBench high-level multi-seed benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDML",
    runner=run_scdml,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDML_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        # Dataset 5's scIB HVG policy is obtained from package metadata:
        # evaluation_batch (batch_key='orig.ident').
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    # scDML's method-specific preprocessing mutates its AnnData input.
    # Isolate each training run from the canonical Dataset 5 reference.
    copy_input_per_seed=True,
)


## 6. Inspect the benchmark results

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  alignment:", run.alignment)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)

    if run.representation_key != "X_scdml":
        raise RuntimeError(f"Unexpected scDML representation for seed {seed}.")
    if run.alignment.get("exact_order_match") is not True:
        raise RuntimeError(f"Seed {seed} failed exact cell barcode alignment.")


## 7. Download the final benchmark archive directly to your computer

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil

archive_path = Path(result.archive_path)
if not archive_path.is_file():
    raise FileNotFoundError(
        f"Final scRareBench archive is missing: {archive_path}"
    )

download_copy = Path("/content/scDML_dataset5_results.zip")
if archive_path.resolve() != download_copy.resolve():
    shutil.copy2(archive_path, download_copy)

with ZipFile(download_copy, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity failure: {bad_member}")

print("Final ZIP:", download_copy)
print("Size (MB):", round(download_copy.stat().st_size / 1024**2, 2))

try:
    from google.colab import files
    files.download(str(download_copy))
except ImportError:
    print("Not running in Colab; ZIP saved to:", download_copy)
